In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, year, month, quarter, weekofyear,
    dayofmonth, date_format, dayofweek, when, explode, sequence
)
from pyspark.sql.window import Window
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_dim_incremental_" + str(datetime.date.today())

# Source silver tables
silver_brand = "retail.silver.ns_brands"
silver_customer = "retail.silver.ns_customers"
silver_item = "retail.silver.ns_items"
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_brand = "retail.gold.dim_brand"
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"


# =========================================================
# AUDIT HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = CAST({rows_read} AS BIGINT),
          rows_written = CAST({rows_written} AS BIGINT),
          rows_rejected = CAST({rows_rejected} AS BIGINT),
          records_before = CAST({records_before} AS BIGINT),
          records_after = CAST({records_after} AS BIGINT)
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = str(error_message).replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def get_unprocessed_silver_run_ids(source_table, gold_table_name):
    return [
        r.etl_run_id
        for r in spark.sql(f"""
            SELECT DISTINCT s.etl_run_id
            FROM {source_table} s
            LEFT ANTI JOIN {processed_runs_table} p
              ON s.etl_run_id = p.source_run_id
             AND p.table_name = '{gold_table_name}'
             AND p.source_layer = 'silver'
             AND p.target_layer = 'gold'
             AND p.status = 'SUCCESS'
            WHERE s.etl_run_id IS NOT NULL
        """).collect()
    ]


def log_layer_processed_runs_gold(source_run_ids, target_run_id, table_name, rows_read, rows_written, rows_rejected):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


def scd2_merge_into_gold(df_tgt, target_table, merge_key, surrogate_key=None):

    temp_view = "vw_gold_incremental_merge"

    target_cols = spark.table(target_table).columns

    # Remove surrogate key
    if surrogate_key:
        merge_cols = [c for c in target_cols if c != surrogate_key]
    else:
        merge_cols = target_cols

    df_tgt = df_tgt.select(*merge_cols)

    # Add SCD2 columns
    df_tgt = (
        df_tgt
        .withColumn("is_current", F.lit(True))
        .withColumn("effective_start_date", F.current_timestamp())
        .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    )

    df_tgt.createOrReplaceTempView(temp_view)

    # =========================================================
    # STEP 1: EXPIRE OLD RECORDS
    # =========================================================
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING {temp_view} AS source
        ON target.{merge_key} = source.{merge_key}
           AND target.is_current = true

        WHEN MATCHED AND target.record_hash <> source.record_hash THEN
          UPDATE SET
            target.is_current = false,
            target.effective_end_date = current_timestamp()
    """)

    # =========================================================
    # STEP 2: INSERT NEW RECORDS
    # =========================================================
    insert_cols = ", ".join(merge_cols)
    insert_vals = ", ".join([f"source.{c}" for c in merge_cols])

    spark.sql(f"""
        INSERT INTO {target_table} ({insert_cols})
        SELECT {insert_vals}
        FROM {temp_view} source
        LEFT JOIN {target_table} target
          ON target.{merge_key} = source.{merge_key}
         AND target.record_hash = source.record_hash
         AND target.is_current = true
        WHERE target.{merge_key} IS NULL
    """)


# =========================================================
# DIM BRAND
# =========================================================
def load_dim_brand_incremental():
    table_name = "dim_brand"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_brand, table_name)
        records_before = spark.table(gold_dim_brand).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_brand}: no new silver runs")
            return

        df_src = spark.table(silver_brand).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("brand_internal_id"),
                "brand_code",
                "brand_name",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("brand_internal_id").orderBy(col("etl_loaded_at").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_brand, "brand_internal_id", "brand_key")

        records_after = spark.table(gold_dim_brand).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_brand}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM CUSTOMER
# =========================================================
def load_dim_customer_incremental():
    table_name = "dim_customer"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_customer, table_name)
        records_before = spark.table(gold_dim_customer).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_customer}: no new silver runs")
            return

        df_src = spark.table(silver_customer).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("customer_internal_id"),
                "entity_id",
                "company_name",
                "customer_type",
                "email",
                "phone",
                "subsidiary",
                "currency_code",
                "terms_name",
                "vat_number",
                "billing_address1",
                "billing_city",
                "billing_state",
                "billing_country",
                "shipping_address1",
                "shipping_city",
                "shipping_state",
                "shipping_country",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("customer_internal_id").orderBy(col("etl_loaded_at").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_customer, "customer_internal_id", "customer_key")

        records_after = spark.table(gold_dim_customer).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_customer}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM ITEM
# =========================================================
def load_dim_item_incremental():
    table_name = "dim_item"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_item, table_name)
        records_before = spark.table(gold_dim_item).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_item}: no new silver runs")
            return

        df_src = spark.table(silver_item).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("item_internal_id"),
                "item_id",
                "item_name",
                "item_type",
                "category",
                "subcategory",
                "brand_internal_id",
                "base_price",
                "cost_price",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("item_internal_id").orderBy(col("etl_loaded_at").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_item, "item_internal_id", "item_key")

        records_after = spark.table(gold_dim_item).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_item}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM DATE - INCREMENTAL INSERT MISSING DATES ONLY
# =========================================================
def load_dim_date_incremental():
    table_name = "dim_date"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        so_run_ids = get_unprocessed_silver_run_ids(silver_sales_orders, table_name)
        sh_run_ids = get_unprocessed_silver_run_ids(silver_shipments, table_name)

        source_run_ids = list(set(so_run_ids + sh_run_ids))
        records_before = spark.table(gold_dim_date).count()

        if not source_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_date}: no new silver runs")
            return

        df_orders = (
            spark.table(silver_sales_orders)
            .filter(col("etl_run_id").isin(so_run_ids))
            .select(col("order_date").alias("dt"))
        )

        df_shipments_1 = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("shipment_date").alias("dt"))
        )

        df_shipments_2 = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("delivery_date").alias("dt"))
        )

        df_dates = (
            df_orders
            .union(df_shipments_1)
            .union(df_shipments_2)
            .filter(col("dt").isNotNull())
        )

        rows_read = df_dates.count()

        if rows_read == 0:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_date}: no valid dates found")
            return

        min_max = df_dates.agg(
            F.min("dt").alias("min_date"),
            F.max("dt").alias("max_date")
        ).collect()[0]

        min_date = min_max["min_date"]
        max_date = min_max["max_date"]

        df_calendar = (
            spark.createDataFrame([(min_date, max_date)], ["start_date", "end_date"])
            .select(explode(sequence(col("start_date"), col("end_date"))).alias("full_date"))
        )

        df_tgt = (
            df_calendar
            .select(
                date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
                col("full_date"),
                dayofmonth(col("full_date")).alias("day_of_month"),
                month(col("full_date")).alias("month_num"),
                date_format(col("full_date"), "MMMM").alias("month_name"),
                quarter(col("full_date")).alias("quarter_num"),
                year(col("full_date")).alias("year_num"),
                weekofyear(col("full_date")).alias("week_of_year"),
                dayofweek(col("full_date")).alias("day_of_week_num"),
                date_format(col("full_date"), "EEEE").alias("day_of_week_name"),
                when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
            )
            .dropDuplicates(["date_key"])
        )

        existing_dates = spark.table(gold_dim_date).select("date_key").dropDuplicates()

        df_new_dates = df_tgt.join(existing_dates, on="date_key", how="left_anti")

        rows_written = df_new_dates.count()

        if rows_written > 0:
            df_new_dates.write.format("delta").mode("append").saveAsTable(gold_dim_date)

        records_after = spark.table(gold_dim_date).count()

        log_layer_processed_runs_gold(source_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_date}: {rows_written} new date rows inserted")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN DIMENSIONS ONLY
# =========================================================
load_dim_brand_incremental()
load_dim_customer_incremental()
load_dim_item_incremental()
load_dim_date_incremental()

## New Incremental for SCD2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, year, month, quarter, weekofyear,
    dayofmonth, date_format, dayofweek, when, explode, sequence
)
from pyspark.sql.window import Window
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_dim_incremental_" + str(datetime.date.today())

# Source silver tables
silver_brand = "retail.silver.ns_brands"
silver_customer = "retail.silver.ns_customers"
silver_item = "retail.silver.ns_items"
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_brand = "retail.gold.dim_brand"
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"


# =========================================================
# AUDIT HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = CAST({rows_read} AS BIGINT),
          rows_written = CAST({rows_written} AS BIGINT),
          rows_rejected = CAST({rows_rejected} AS BIGINT),
          records_before = CAST({records_before} AS BIGINT),
          records_after = CAST({records_after} AS BIGINT)
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = str(error_message).replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def get_unprocessed_silver_run_ids(source_table, gold_table_name):
    return [
        r.etl_run_id
        for r in spark.sql(f"""
            SELECT DISTINCT s.etl_run_id
            FROM {source_table} s
            LEFT ANTI JOIN {processed_runs_table} p
              ON s.etl_run_id = p.source_run_id
             AND p.table_name = '{gold_table_name}'
             AND p.source_layer = 'silver'
             AND p.target_layer = 'gold'
             AND p.status = 'SUCCESS'
            WHERE s.etl_run_id IS NOT NULL
        """).collect()
    ]


def log_layer_processed_runs_gold(source_run_ids, target_run_id, table_name, rows_read, rows_written, rows_rejected):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


def scd2_merge_into_gold(df_tgt, target_table, merge_key, surrogate_key=None):
    """
    SCD Type 2 merge pattern:
    1) Expire current target record when source hash changed.
    2) Insert new version for new keys or changed hashes.

    Assumption: target surrogate key is generated by Delta identity column.
    """
    temp_view = "vw_gold_incremental_merge"
    target_cols = spark.table(target_table).columns

    if surrogate_key:
        insert_cols = [c for c in target_cols if c != surrogate_key]
    else:
        insert_cols = target_cols

    df_tgt = (
        df_tgt
        .withColumn("is_current", F.lit(True))
        .withColumn(
            "effective_start_date",
            F.coalesce(col("etl_loaded_at").cast("timestamp"), F.current_timestamp())
        )
        .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    )

    df_tgt = df_tgt.select(*insert_cols)
    df_tgt.createOrReplaceTempView(temp_view)

    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING {temp_view} AS source
        ON target.{merge_key} = source.{merge_key}
           AND target.is_current = true

        WHEN MATCHED AND target.record_hash <> source.record_hash THEN
          UPDATE SET
            target.is_current = false,
            target.effective_end_date = source.effective_start_date
    """)

    insert_col_sql = ", ".join(insert_cols)
    insert_val_sql = ", ".join([f"source.{c}" for c in insert_cols])

    spark.sql(f"""
        INSERT INTO {target_table} ({insert_col_sql})
        SELECT {insert_val_sql}
        FROM {temp_view} source
        LEFT JOIN {target_table} target
          ON target.{merge_key} = source.{merge_key}
         AND target.record_hash = source.record_hash
         AND target.is_current = true
        WHERE target.{merge_key} IS NULL
    """)


# =========================================================
# DIM BRAND
# =========================================================
def load_dim_brand_incremental():
    table_name = "dim_brand"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_brand, table_name)
        records_before = spark.table(gold_dim_brand).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_brand}: no new silver runs")
            return

        df_src = spark.table(silver_brand).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("brand_internal_id"),
                "brand_code",
                "brand_name",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("brand_internal_id").orderBy(col("etl_loaded_at").desc(), col("record_hash").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_brand, "brand_internal_id", "brand_key")

        records_after = spark.table(gold_dim_brand).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_brand}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM CUSTOMER
# =========================================================
def load_dim_customer_incremental():
    table_name = "dim_customer"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_customer, table_name)
        records_before = spark.table(gold_dim_customer).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_customer}: no new silver runs")
            return

        df_src = spark.table(silver_customer).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("customer_internal_id"),
                "entity_id",
                "company_name",
                "customer_type",
                "email",
                "phone",
                "subsidiary",
                "currency_code",
                "terms_name",
                "vat_number",
                "billing_address1",
                "billing_city",
                "billing_state",
                "billing_country",
                "shipping_address1",
                "shipping_city",
                "shipping_state",
                "shipping_country",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("customer_internal_id").orderBy(col("etl_loaded_at").desc(), col("record_hash").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_customer, "customer_internal_id", "customer_key")

        records_after = spark.table(gold_dim_customer).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_customer}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM ITEM
# =========================================================
def load_dim_item_incremental():
    table_name = "dim_item"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        silver_run_ids = get_unprocessed_silver_run_ids(silver_item, table_name)
        records_before = spark.table(gold_dim_item).count()

        if not silver_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_item}: no new silver runs")
            return

        df_src = spark.table(silver_item).filter(col("etl_run_id").isin(silver_run_ids))
        rows_read = df_src.count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("item_internal_id"),
                "item_id",
                "item_name",
                "item_type",
                "category",
                "subcategory",
                "brand_internal_id",
                "base_price",
                "cost_price",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
        )

        window_spec = Window.partitionBy("item_internal_id").orderBy(col("etl_loaded_at").desc(), col("record_hash").desc())

        df_tgt = (
            df_tgt
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        rows_written = df_tgt.count()

        scd2_merge_into_gold(df_tgt, gold_dim_item, "item_internal_id", "item_key")

        records_after = spark.table(gold_dim_item).count()

        log_layer_processed_runs_gold(silver_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_item}: incremental MERGE completed")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM DATE - INCREMENTAL INSERT MISSING DATES ONLY
# =========================================================
def load_dim_date_incremental():
    table_name = "dim_date"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        so_run_ids = get_unprocessed_silver_run_ids(silver_sales_orders, table_name)
        sh_run_ids = get_unprocessed_silver_run_ids(silver_shipments, table_name)

        source_run_ids = list(set(so_run_ids + sh_run_ids))
        records_before = spark.table(gold_dim_date).count()

        if not source_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_date}: no new silver runs")
            return

        df_orders = (
            spark.table(silver_sales_orders)
            .filter(col("etl_run_id").isin(so_run_ids))
            .select(col("order_date").alias("dt"))
        )

        df_shipments_1 = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("shipment_date").alias("dt"))
        )

        df_shipments_2 = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("delivery_date").alias("dt"))
        )

        df_dates = (
            df_orders
            .union(df_shipments_1)
            .union(df_shipments_2)
            .filter(col("dt").isNotNull())
        )

        rows_read = df_dates.count()

        if rows_read == 0:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_dim_date}: no valid dates found")
            return

        min_max = df_dates.agg(
            F.min("dt").alias("min_date"),
            F.max("dt").alias("max_date")
        ).collect()[0]

        min_date = min_max["min_date"]
        max_date = min_max["max_date"]

        df_calendar = (
            spark.createDataFrame([(min_date, max_date)], ["start_date", "end_date"])
            .select(explode(sequence(col("start_date"), col("end_date"))).alias("full_date"))
        )

        df_tgt = (
            df_calendar
            .select(
                date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
                col("full_date"),
                dayofmonth(col("full_date")).alias("day_of_month"),
                month(col("full_date")).alias("month_num"),
                date_format(col("full_date"), "MMMM").alias("month_name"),
                quarter(col("full_date")).alias("quarter_num"),
                year(col("full_date")).alias("year_num"),
                weekofyear(col("full_date")).alias("week_of_year"),
                dayofweek(col("full_date")).alias("day_of_week_num"),
                date_format(col("full_date"), "EEEE").alias("day_of_week_name"),
                when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
            )
            .dropDuplicates(["date_key"])
        )

        existing_dates = spark.table(gold_dim_date).select("date_key").dropDuplicates()

        df_new_dates = df_tgt.join(existing_dates, on="date_key", how="left_anti")

        rows_written = df_new_dates.count()

        if rows_written > 0:
            df_new_dates.write.format("delta").mode("append").saveAsTable(gold_dim_date)

        records_after = spark.table(gold_dim_date).count()

        log_layer_processed_runs_gold(source_run_ids, run_id, table_name, rows_read, rows_written, 0)
        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_date}: {rows_written} new date rows inserted")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN DIMENSIONS ONLY
# =========================================================
load_dim_brand_incremental()
load_dim_customer_incremental()
load_dim_item_incremental()
load_dim_date_incremental()